# coerce-float-arg-to-array — ex2: coerce_args: apply scalar coercion across an args tuple, ndarray pass-through

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `coerce-float-arg-to-array`. Running the final beacon cell reports progress against the `Backprop: Coerce float arg to array` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """Minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries optional `.recipe`,
    `.requires_grad`, and `.grad` (the accumulated gradient at leaves)."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Coerce float arg to array` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`coerce-float-arg-to-array`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "coerce-float-arg-to-array"
DD_SUBTOPIC = "Backprop: Coerce float arg to array"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Coerce-args — apply scalar coercion across an args tuple — refresher

Ex1's `coerce_to_array` handles a SINGLE arg. The wrapper actually calls it across the whole positional-args tuple:

```python
args_coerced = tuple(coerce_to_array(a) for a in args)
```

Same rules per arg:
- `bool` → pass-through (the subclass-of-int trap)
- `int` / `float` → `t.tensor(float(arg))`
- everything else → pass-through (including `torch.Tensor`, `np.ndarray`, `MiniTensor`, tuples, None, ...)

Critical pass-through case: `np.ndarray`. We DON'T want to wrap it in `t.tensor(...)` here — the downstream unbox step is doing the MiniTensor.array extraction, and a numpy array is already raw-array-shaped. (The actual raw fn — `torch.log` — accepts numpy arrays via the tensor protocol.)

### Exercise 2 — coerce_args: apply scalar coercion across an args tuple, ndarray pass-through

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the per-arg scalar coercion across a whole args tuple, preserving non-scalar types — especially numpy ndarrays — as pass-through so the downstream unbox step receives uniform input.
> Keywords: coerce, args-tuple, ndarray, pass-through, wrap-forward
> ```

**KCs targeted:** `coerce-float-arg-to-array`, `unbox-args-tensor-to-array`

Implement `coerce_args(args)`. Given a positional-args tuple, return a new tuple where each arg has been passed through the single-arg coercion rule:

- `bool` (incl. `True` / `False`) → pass-through (NOT a tensor).
- `int` or `float` (excluding bool) → `t.tensor(float(arg))`.
- Everything else (`torch.Tensor`, `MiniTensor`, `np.ndarray`, tuples, `None`, strings) → pass-through unchanged (identity).

Examples:

```
coerce_args((m, 3.0, 5))    → (m, tensor(3.0), tensor(5.0))
coerce_args((arr, True))    → (arr, True)             # ndarray + bool pass-through
coerce_args(())             → ()
```

**The load-bearing invariants for ex2** (ex1 covered the per-arg rule):

1. **`np.ndarray` MUST pass through.** Numpy arrays are already raw-array-shaped; the downstream unbox step doesn't need to convert them, and the raw torch fn accepts numpy via the tensor protocol. Wrapping it in `t.tensor(arr)` would copy memory and promote dtype.

2. **A list of plain ints is pass-through too** — only `int` and `float` themselves (not collections containing them) coerce. (Lists go to nested-unbox, not coerce.)

3. **Order + length preserved.**

You may call a single-arg helper `_coerce_one` inside. Or inline the rule. Either is fine.

In [ ]:
def coerce_args(args: tuple) -> tuple:
    """Apply scalar coercion (int/float → 0-D tensor) per-arg; pass everything else through."""
    raise NotImplementedError()


def _test_ex2():
    # --- empty + all pass-through ---
    assert coerce_args(()) == ()
    result = coerce_args(('x', None, (3, 4)))
    assert result == ('x', None, (3, 4))

    # --- scalar coercion across mixed tuple ---
    raw = t.tensor([1.0, 2.0, 3.0])
    result = coerce_args((raw, 3.0, 5))
    assert len(result) == 3
    assert result[0] is raw, 'raw tensor identity preserved'
    assert isinstance(result[1], t.Tensor)
    assert result[1].ndim == 0 and result[1].item() == 3.0
    assert isinstance(result[2], t.Tensor)
    assert result[2].dtype == t.float32, 'int → float tensor (not int tensor)'
    assert result[2].item() == 5.0

    # --- bool pass-through (subclass-of-int trap, from ex1) ---
    result = coerce_args((True, False, 2.5))
    assert result[0] is True, 'True must pass through, NOT become tensor(1.0)'
    assert result[1] is False
    assert isinstance(result[2], t.Tensor) and result[2].item() == 2.5

    # --- numpy ndarray MUST pass through (NEW invariant for ex2) ---
    arr = np.array([1.0, 2.0, 3.0])
    result = coerce_args((arr,))
    assert result[0] is arr, (
        'np.ndarray must pass through (NOT be wrapped in t.tensor) — '
        'downstream unbox handles raw-array types; wrapping would copy memory'
    )

    # --- np.ndarray + tensor + float in one call ---
    result = coerce_args((arr, raw, 1.5))
    assert result[0] is arr
    assert result[1] is raw
    assert isinstance(result[2], t.Tensor) and result[2].item() == 1.5

    # --- MiniTensor pass-through (the wrapper unboxes it later, not here) ---
    mt = MiniTensor(t.tensor([1.0]))
    result = coerce_args((mt, 3.0))
    assert result[0] is mt, 'MiniTensor unchanged at coerce step (unbox happens after)'

    # --- list of plain ints passes through (lists aren't scalars) ---
    result = coerce_args(([1, 2, 3], (4, 5)))
    assert result == ([1, 2, 3], (4, 5))
    assert isinstance(result[0], list)

    # --- length and order always preserved ---
    for inp in [(1,), (1, 2, 3), (1.0, 2.0, 'x'), (True, 1, None)]:
        out = coerce_args(inp)
        assert len(out) == len(inp), f'length: {inp} → {out}'

    # --- the result is still a tuple, not a generator/list ---
    result = coerce_args((1, 2))
    assert isinstance(result, tuple), f'must return tuple, got {type(result).__name__}'
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def coerce_args(args: tuple) -> tuple:
    def _coerce_one(a):
        # bool first (subclass-of-int trap): must pass through.
        if isinstance(a, bool):
            return a
        if isinstance(a, (int, float)):
            return t.tensor(float(a))
        # Everything else (Tensor, MiniTensor, ndarray, list, tuple,
        # None, str, ...) is pass-through — the downstream unbox /
        # nested-unbox step handles array-shaped objects.
        return a
    return tuple(_coerce_one(a) for a in args)
```

**Why ndarray is pass-through (not wrapped).** The raw forward fn (`torch.log`, `torch.add`) accepts numpy arrays — PyTorch's tensor protocol handles the conversion at the C++ layer with zero copy on CPU. Pre-emptively calling `t.tensor(arr)` would (a) trigger a copy + dtype-promotion in Python, and (b) break the identity invariant the Recipe relies on (different object stored from what the user passed).

**Per-arg vs. whole-tuple.** Ex1 was the single-arg helper. Ex2 is the tuple version. The wrapper actually calls the tuple version — ex1 is the building block, ex2 is the production callsite. Same rule, applied N times.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()